# Text-to-Sign Generation Design Notebook

## Notebook 08: Doctor Text/Speech to KSL Gloss and Sign/Avatar Output

This notebook starts the **reverse translation direction** of the project.

So far, the project has focused mainly on:

```text
KSL/sign video → recognized sign/gloss → healthcare text
```

Now we design the opposite direction:

```text
doctor text/speech → healthcare text → KSL gloss → sign/avatar output
```

This is important because the final system should be **bidirectional**.

---

# Full bidirectional project goal

The final web application should support two communication flows:

## Flow A: Deaf/KSL user to healthcare worker

```text
Camera/video input
→ MediaPipe landmark extraction
→ sign recognition model
→ recognized gloss
→ healthcare text
→ optional speech output
```

## Flow B: Healthcare worker to deaf/KSL user

```text
doctor speech/text
→ speech-to-text if voice
→ healthcare text normalization
→ text-to-KSL gloss
→ sign asset/avatar/diffusion generation
→ visual KSL output
```

This notebook focuses on **Flow B**.

---

# Models/components used in this notebook

This notebook uses three levels of modelling/design.

| Level | Component | Purpose |
|---|---|---|
| 1 | Rule-based text-to-gloss mapper | Converts known healthcare sentences to draft KSL-style gloss |
| 2 | Sign asset/avatar selector | Maps gloss tokens or phrases to video/avatar placeholders |
| 3 | Future diffusion generation design | Documents how a future model can generate sign motion/avatar output |

## Current practical baseline

For now, the practical baseline is:

```text
healthcare text → phrase mapping → KSL-style gloss → sign asset sequence
```

This is realistic for an MVP web app.

## Future research model

Later, the advanced model can be:

```text
healthcare text → mT5/T5 text-to-gloss → diffusion-based sign motion generation → avatar
```

This notebook prepares that design.

## 1. Import libraries

This notebook mainly uses Python table/data utilities.

It does not train a heavy diffusion model yet.

Instead, it creates the text-to-sign design artifacts needed for implementation and thesis documentation.

In [1]:
import json
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

print("Text-to-sign generation design notebook started successfully")

Text-to-sign generation design notebook started successfully


## 2. Configure project folders

The notebook saves outputs under:

```text
/workspace/dataset/ksl_project_data/text_to_sign/
```

If you are running outside Docker, it falls back to:

```text
ksl_project_data/text_to_sign/
```

In [2]:
PROJECT_DIR = Path("/workspace/dataset/ksl_project_data").resolve()

if not Path("/workspace/dataset").exists():
    PROJECT_DIR = Path("ksl_project_data").resolve()

TEXT_TO_SIGN_DIR = PROJECT_DIR / "text_to_sign"
LANGUAGE_DIR = PROJECT_DIR / "language_translation"
VALIDATION_DIR = PROJECT_DIR / "validation"
MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [PROJECT_DIR, TEXT_TO_SIGN_DIR, LANGUAGE_DIR, VALIDATION_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Text-to-sign directory:", TEXT_TO_SIGN_DIR)

Project directory: /workspace/dataset/ksl_project_data
Text-to-sign directory: /workspace/dataset/ksl_project_data/text_to_sign


## 3. Load healthcare phrase dataset from Notebook 06

Notebook 06 created:

```text
language_translation/healthcare_gloss_phrase_dataset.csv
```

This file contains phrase pairs such as:

```text
gloss: pain where
text: Where is the pain?
```

For text-to-sign, we reverse the direction:

```text
healthcare text → gloss
```

In [3]:
phrase_dataset_path = LANGUAGE_DIR / "healthcare_gloss_phrase_dataset.csv"

if phrase_dataset_path.exists():
    phrase_df = pd.read_csv(phrase_dataset_path)
    print("Loaded phrase dataset:", phrase_dataset_path)
else:
    print("Phrase dataset not found. Creating fallback phrases.")
    phrase_df = pd.DataFrame([
        {"id": 1, "domain": "triage", "gloss": "pain where", "healthcare_text": "Where is the pain?", "priority": "critical"},
        {"id": 2, "domain": "emergency", "gloss": "you chest pain have", "healthcare_text": "Do you have chest pain?", "priority": "critical"},
        {"id": 3, "domain": "medicine", "gloss": "medicine allergy you", "healthcare_text": "Are you allergic to any medicine?", "priority": "critical"},
        {"id": 4, "domain": "pharmacy", "gloss": "medicine take day two", "healthcare_text": "Take this medicine twice a day.", "priority": "important"},
        {"id": 5, "domain": "follow_up", "gloss": "come back week one", "healthcare_text": "Come back after one week.", "priority": "normal"},
        {"id": 6, "domain": "reception", "gloss": "doctor wait please", "healthcare_text": "Please wait for the doctor.", "priority": "normal"},
        {"id": 7, "domain": "emergency", "gloss": "breathing difficult you", "healthcare_text": "Do you have difficulty breathing?", "priority": "critical"},
        {"id": 8, "domain": "assessment", "gloss": "pregnant you", "healthcare_text": "Are you pregnant?", "priority": "important"},
        {"id": 9, "domain": "medicine", "gloss": "medicine today take you", "healthcare_text": "Have you taken medicine today?", "priority": "important"},
        {"id": 10, "domain": "diagnosis", "gloss": "hurt where show me", "healthcare_text": "Show me where it hurts.", "priority": "critical"},
    ])

phrase_df

Loaded phrase dataset: /workspace/dataset/ksl_project_data/language_translation/healthcare_gloss_phrase_dataset.csv


,id,domain,gloss,healthcare_text,priority
0,1,triage,pain where,Where is the pain?,critical
1,2,emergency,you chest pain have,Do you have chest pain?,critical
2,3,medicine,medicine allergy you,Are you allergic to any medicine?,critical
3,4,pharmacy,medicine take day two,Take this medicine twice a day.,important
4,5,follow_up,come back week one,Come back after one week.,normal
5,6,reception,doctor wait please,Please wait for the doctor.,normal
6,7,emergency,breathing difficult you,Do you have difficulty breathing?,critical
7,8,assessment,pregnant you,Are you pregnant?,important
8,9,medicine,medicine today take you,Have you taken medicine today?,important
9,10,diagnosis,hurt where show me,Show me where it hurts.,critical


## 4. Normalize healthcare text

Doctors may type phrases in different ways:

```text
Where is the pain?
where is pain
WHERE IS THE PAIN
```

This function normalizes text so that phrase matching becomes easier.

In [4]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


phrase_df["normalized_text"] = phrase_df["healthcare_text"].apply(normalize_text)
phrase_df["normalized_gloss"] = phrase_df["gloss"].apply(normalize_text)

phrase_df[["healthcare_text", "normalized_text", "gloss"]].head()

,healthcare_text,normalized_text,gloss
0,Where is the pain?,where is the pain,pain where
1,Do you have chest pain?,do you have chest pain,you chest pain have
2,Are you allergic to any medicine?,are you allergic to any medicine,medicine allergy you
3,Take this medicine twice a day.,take this medicine twice a day,medicine take day two
4,Come back after one week.,come back after one week,come back week one


## 5. Model 1: Exact text-to-gloss mapper

This is the first baseline model.

It maps a known healthcare sentence directly to a draft KSL-style gloss.

Example:

```text
Input text:
Where is the pain?

Output gloss:
pain where
```

## Strength

- Very reliable for known healthcare phrases.
- Easy to validate.
- Good for a controlled MVP.

## Weakness

- Does not generalize well to new sentences.
- Needs a phrase dictionary.
- Must be reviewed by KSL experts.

In [5]:
class ExactTextToGlossMapper:
    def __init__(self, phrase_dataframe):
        self.mapping = {}

        for _, row in phrase_dataframe.iterrows():
            text_key = normalize_text(row["healthcare_text"])

            self.mapping[text_key] = {
                "gloss": row["gloss"],
                "domain": row["domain"],
                "priority": row["priority"],
                "phrase_id": int(row["id"]),
            }

    def translate(self, healthcare_text):
        text_key = normalize_text(healthcare_text)

        if text_key not in self.mapping:
            return {
                "gloss": "UNKNOWN_PHRASE",
                "domain": None,
                "priority": None,
                "phrase_id": None,
                "match_type": "no_exact_match",
            }

        result = self.mapping[text_key].copy()
        result["match_type"] = "exact"
        return result


exact_text_to_gloss = ExactTextToGlossMapper(phrase_df)

test_texts = [
    "Where is the pain?",
    "Take this medicine twice a day.",
    "Please wait for the doctor.",
    "This is not in the dictionary.",
]

for text in test_texts:
    result = exact_text_to_gloss.translate(text)
    print("Input:", text)
    print("Gloss:", result["gloss"])
    print("Domain:", result["domain"])
    print("Priority:", result["priority"])
    print("Match:", result["match_type"])
    print("-" * 80)

Input: Where is the pain?
Gloss: pain where
Domain: triage
Priority: critical
Match: exact
--------------------------------------------------------------------------------
Input: Take this medicine twice a day.
Gloss: medicine take day two
Domain: pharmacy
Priority: important
Match: exact
--------------------------------------------------------------------------------
Input: Please wait for the doctor.
Gloss: doctor wait please
Domain: reception
Priority: normal
Match: exact
--------------------------------------------------------------------------------
Input: This is not in the dictionary.
Gloss: UNKNOWN_PHRASE
Domain: None
Priority: None
Match: no_exact_match
--------------------------------------------------------------------------------


## 6. Model 2: Fuzzy text-to-gloss mapper

Exact matching is too strict.

A doctor may type:

```text
Do you feel chest pain?
```

while the phrase dataset contains:

```text
Do you have chest pain?
```

The fuzzy mapper uses token overlap to find the closest known healthcare phrase.

In [6]:
def token_set(text):
    return set(normalize_text(text).split())


def jaccard_similarity(text_a, text_b):
    set_a = token_set(text_a)
    set_b = token_set(text_b)

    if not set_a and not set_b:
        return 1.0

    if not set_a or not set_b:
        return 0.0

    return len(set_a.intersection(set_b)) / len(set_a.union(set_b))


class FuzzyTextToGlossMapper:
    def __init__(self, phrase_dataframe, threshold=0.45):
        self.phrase_dataframe = phrase_dataframe.copy()
        self.threshold = threshold

    def translate(self, healthcare_text):
        best_score = -1
        best_row = None

        for _, row in self.phrase_dataframe.iterrows():
            score = jaccard_similarity(healthcare_text, row["healthcare_text"])

            if score > best_score:
                best_score = score
                best_row = row

        if best_score < self.threshold:
            return {
                "gloss": "UNKNOWN_PHRASE",
                "matched_text": None,
                "score": best_score,
                "domain": None,
                "priority": None,
                "phrase_id": None,
                "match_type": "no_fuzzy_match",
            }

        return {
            "gloss": best_row["gloss"],
            "matched_text": best_row["healthcare_text"],
            "score": best_score,
            "domain": best_row["domain"],
            "priority": best_row["priority"],
            "phrase_id": int(best_row["id"]),
            "match_type": "fuzzy",
        }


fuzzy_text_to_gloss = FuzzyTextToGlossMapper(phrase_df, threshold=0.35)

test_texts = [
    "Where do you feel pain?",
    "Do you feel chest pain?",
    "Are you allergic to medicine?",
    "Take medicine two times per day.",
    "Please wait for doctor.",
    "Random unrelated sentence.",
]

for text in test_texts:
    result = fuzzy_text_to_gloss.translate(text)

    print("Input:", text)
    print("Matched text:", result["matched_text"])
    print("Gloss:", result["gloss"])
    print("Score:", round(result["score"], 3))
    print("Domain:", result["domain"])
    print("Priority:", result["priority"])
    print("-" * 80)

Input: Where do you feel pain?
Matched text: Do you have chest pain?
Gloss: you chest pain have
Score: 0.429
Domain: emergency
Priority: critical
--------------------------------------------------------------------------------
Input: Do you feel chest pain?
Matched text: Do you have chest pain?
Gloss: you chest pain have
Score: 0.667
Domain: emergency
Priority: critical
--------------------------------------------------------------------------------
Input: Are you allergic to medicine?
Matched text: Are you allergic to any medicine?
Gloss: medicine allergy you
Score: 0.833
Domain: medicine
Priority: critical
--------------------------------------------------------------------------------
Input: Take medicine two times per day.
Matched text: None
Gloss: UNKNOWN_PHRASE
Score: 0.333
Domain: None
Priority: None
--------------------------------------------------------------------------------
Input: Please wait for doctor.
Matched text: Please wait for the doctor.
Gloss: doctor wait please
S

## 7. Create sign asset dictionary

For a practical MVP web app, do not start with full AI avatar generation immediately.

Start with a sign asset dictionary.

The asset dictionary maps each gloss token to a placeholder video/avatar asset.

Example:

```text
pain → assets/signs/pain.mp4
where → assets/signs/where.mp4
```

Then a phrase can be displayed as a sequence of sign clips.

This is not perfect natural KSL, but it is a realistic prototype step.

In [7]:
# Collect unique gloss tokens from phrase dataset
all_tokens = []

for gloss in phrase_df["gloss"]:
    all_tokens.extend(normalize_text(gloss).split())

unique_tokens = sorted(set(all_tokens))

asset_rows = []

for token in unique_tokens:
    asset_rows.append(
        {
            "gloss_token": token,
            "asset_type": "video_placeholder",
            "asset_path": f"assets/signs/{token}.mp4",
            "avatar_motion_path": f"assets/avatar_motions/{token}.json",
            "availability_status": "placeholder_needed",
            "review_status": "pending",
        }
    )

asset_df = pd.DataFrame(asset_rows)
asset_df.head(20)

,gloss_token,asset_type,asset_path,avatar_motion_path,availability_status,review_status
0,admit,video_placeholder,assets/signs/admit.mp4,assets/avatar_motions/admit.json,placeholder_needed,pending
1,after,video_placeholder,assets/signs/after.mp4,assets/avatar_motions/after.json,placeholder_needed,pending
2,age,video_placeholder,assets/signs/age.mp4,assets/avatar_motions/age.json,placeholder_needed,pending
3,allergy,video_placeholder,assets/signs/allergy.mp4,assets/avatar_motions/allergy.json,placeholder_needed,pending
4,ambulance,video_placeholder,assets/signs/ambulance.mp4,assets/avatar_motions/ambulance.json,placeholder_needed,pending
5,back,video_placeholder,assets/signs/back.mp4,assets/avatar_motions/back.json,placeholder_needed,pending
6,before,video_placeholder,assets/signs/before.mp4,assets/avatar_motions/before.json,placeholder_needed,pending
7,blood,video_placeholder,assets/signs/blood.mp4,assets/avatar_motions/blood.json,placeholder_needed,pending
8,breathing,video_placeholder,assets/signs/breathing.mp4,assets/avatar_motions/breathing.json,placeholder_needed,pending
9,call,video_placeholder,assets/signs/call.mp4,assets/avatar_motions/call.json,placeholder_needed,pending


## 8. Save sign asset dictionary

This file tells the future web app which sign asset to play for each gloss token.

Later, you can replace placeholders with real:

- KSL sign videos
- avatar animations
- motion capture files
- diffusion-generated motion files

In [8]:
asset_dictionary_path = TEXT_TO_SIGN_DIR / "sign_asset_dictionary.csv"
asset_df.to_csv(asset_dictionary_path, index=False)

print("Saved sign asset dictionary:", asset_dictionary_path)

Saved sign asset dictionary: /workspace/dataset/ksl_project_data/text_to_sign/sign_asset_dictionary.csv


## 9. Convert gloss to sign asset sequence

This function receives a gloss phrase and returns the ordered sign assets.

Example:

```text
gloss: pain where
assets:
1. assets/signs/pain.mp4
2. assets/signs/where.mp4
```

This is the simplest visual sign output design.

In [9]:
asset_lookup = {
    row["gloss_token"]: row.to_dict()
    for _, row in asset_df.iterrows()
}


def gloss_to_asset_sequence(gloss):
    tokens = normalize_text(gloss).split()

    sequence = []

    for index, token in enumerate(tokens, start=1):
        if token in asset_lookup:
            asset = asset_lookup[token]
            sequence.append(
                {
                    "order": index,
                    "gloss_token": token,
                    "asset_path": asset["asset_path"],
                    "avatar_motion_path": asset["avatar_motion_path"],
                    "availability_status": asset["availability_status"],
                }
            )
        else:
            sequence.append(
                {
                    "order": index,
                    "gloss_token": token,
                    "asset_path": None,
                    "avatar_motion_path": None,
                    "availability_status": "missing_token",
                }
            )

    return sequence


example_gloss = "medicine take day two"
asset_sequence = gloss_to_asset_sequence(example_gloss)

pd.DataFrame(asset_sequence)

,order,gloss_token,asset_path,avatar_motion_path,availability_status
0,1,medicine,assets/signs/medicine.mp4,assets/avatar_motions/medicine.json,placeholder_needed
1,2,take,assets/signs/take.mp4,assets/avatar_motions/take.json,placeholder_needed
2,3,day,assets/signs/day.mp4,assets/avatar_motions/day.json,placeholder_needed
3,4,two,assets/signs/two.mp4,assets/avatar_motions/two.json,placeholder_needed


## 10. End-to-end text-to-sign baseline function

This combines:

```text
healthcare text
→ text-to-gloss mapping
→ sign asset sequence
```

This is what the future web app backend can expose as an API endpoint.

Example API behavior:

```text
POST /api/text-to-sign/
Input: "Take this medicine twice a day."
Output:
{
  "gloss": "medicine take day two",
  "assets": [...]
}
```

In [10]:
def text_to_sign_pipeline(healthcare_text, use_fuzzy=True):
    exact_result = exact_text_to_gloss.translate(healthcare_text)

    if exact_result["gloss"] != "UNKNOWN_PHRASE":
        mapping_result = exact_result
        matched_text = healthcare_text
        score = 1.0
    elif use_fuzzy:
        fuzzy_result = fuzzy_text_to_gloss.translate(healthcare_text)
        mapping_result = fuzzy_result
        matched_text = fuzzy_result.get("matched_text")
        score = fuzzy_result.get("score")
    else:
        mapping_result = exact_result
        matched_text = None
        score = 0.0

    gloss = mapping_result["gloss"]

    if gloss == "UNKNOWN_PHRASE":
        assets = []
    else:
        assets = gloss_to_asset_sequence(gloss)

    return {
        "input_text": healthcare_text,
        "matched_text": matched_text,
        "gloss": gloss,
        "domain": mapping_result.get("domain"),
        "priority": mapping_result.get("priority"),
        "phrase_id": mapping_result.get("phrase_id"),
        "match_type": mapping_result.get("match_type"),
        "score": score,
        "sign_assets": assets,
    }


demo_inputs = [
    "Where is the pain?",
    "Do you feel chest pain?",
    "Are you allergic to medicine?",
    "Take this medicine twice a day.",
    "Please wait for doctor.",
    "Unknown sentence not available.",
]

demo_outputs = []

for text in demo_inputs:
    output = text_to_sign_pipeline(text)
    demo_outputs.append(output)

    print(json.dumps(output, indent=2))
    print("=" * 100)

{
  "input_text": "Where is the pain?",
  "matched_text": "Where is the pain?",
  "gloss": "pain where",
  "domain": "triage",
  "priority": "critical",
  "phrase_id": 1,
  "match_type": "exact",
  "score": 1.0,
  "sign_assets": [
    {
      "order": 1,
      "gloss_token": "pain",
      "asset_path": "assets/signs/pain.mp4",
      "avatar_motion_path": "assets/avatar_motions/pain.json",
      "availability_status": "placeholder_needed"
    },
    {
      "order": 2,
      "gloss_token": "where",
      "asset_path": "assets/signs/where.mp4",
      "avatar_motion_path": "assets/avatar_motions/where.json",
      "availability_status": "placeholder_needed"
    }
  ]
}
{
  "input_text": "Do you feel chest pain?",
  "matched_text": "Do you have chest pain?",
  "gloss": "you chest pain have",
  "domain": "emergency",
  "priority": "critical",
  "phrase_id": 2,
  "match_type": "fuzzy",
  "score": 0.6666666666666666,
  "sign_assets": [
    {
      "order": 1,
      "gloss_token": "you",
     

## 11. Save demo text-to-sign outputs

These outputs can be used as examples for the web app API design.

In [12]:
demo_output_path = TEXT_TO_SIGN_DIR / "text_to_sign_demo_outputs.json"
demo_output_path.write_text(json.dumps(demo_outputs, indent=2), encoding="utf-8")

print("Saved demo outputs:", demo_output_path)

Saved demo outputs: /workspace/dataset/ksl_project_data/text_to_sign/text_to_sign_demo_outputs.json


## 12. Prepare future model training format

For future T5/mT5 training, we need examples in this form:

```text
input: convert healthcare text to ksl gloss: Where is the pain?
target: pain where
```

This is the reverse of Notebook 06.

In [13]:
text_to_gloss_rows = []

for _, row in phrase_df.iterrows():
    text_to_gloss_rows.append(
        {
            "input_text": f"convert healthcare text to ksl gloss: {row['healthcare_text']}",
            "target_gloss": row["gloss"],
            "domain": row["domain"],
            "priority": row["priority"],
        }
    )

text_to_gloss_df = pd.DataFrame(text_to_gloss_rows)

text_to_gloss_dataset_path = TEXT_TO_SIGN_DIR / "text_to_gloss_training_dataset.csv"
text_to_gloss_df.to_csv(text_to_gloss_dataset_path, index=False)

print("Saved text-to-gloss training dataset:", text_to_gloss_dataset_path)
text_to_gloss_df.head()

Saved text-to-gloss training dataset: /workspace/dataset/ksl_project_data/text_to_sign/text_to_gloss_training_dataset.csv


,input_text,target_gloss,domain,priority
0,convert healthcare text to ksl gloss: Where is...,pain where,triage,critical
1,convert healthcare text to ksl gloss: Do you h...,you chest pain have,emergency,critical
2,convert healthcare text to ksl gloss: Are you ...,medicine allergy you,medicine,critical
3,convert healthcare text to ksl gloss: Take thi...,medicine take day two,pharmacy,important
4,convert healthcare text to ksl gloss: Come bac...,come back week one,follow_up,normal


## 13. Future diffusion/avatar generation design

A full diffusion-based sign generation model would need more data than we currently have.

The future design can be:

```text
healthcare text
→ mT5 text-to-gloss model
→ gloss sequence
→ sign motion representation
→ diffusion model generates motion sequence
→ avatar renders sign animation
```

## What the diffusion model would generate

It would not generate normal text.

It would generate sign motion data such as:

```text
time × joints × coordinates
```

Example shape:

```text
60 frames × 75 joints × 3 coordinates
```

That output can be rendered using a 3D avatar or skeleton viewer.

In [14]:
future_diffusion_design = {
    "model_name": "Future diffusion-based sign motion generator",
    "current_status": "design_only_not_trained",
    "reason_not_trained_now": "Requires validated KSL motion dataset, signer diversity, and more compute.",
    "input": {
        "type": "KSL gloss sequence or healthcare text embedding",
        "example": "medicine take day two",
    },
    "output": {
        "type": "sign motion sequence",
        "shape_example": "60_frames x 75_joints x 3_coordinates",
        "render_target": "2D skeleton, 3D avatar, or sign video animation",
    },
    "future_training_data_needed": [
        "validated KSL healthcare phrase videos",
        "frame-level pose and hand landmarks",
        "gloss labels",
        "healthcare text labels",
        "multiple signers",
        "lighting/background variation",
    ],
    "future_model_pipeline": [
        "text_to_gloss_model",
        "gloss_embedding",
        "conditional_diffusion_motion_generator",
        "motion_smoothing",
        "avatar_renderer",
    ],
}

future_diffusion_design_path = TEXT_TO_SIGN_DIR / "future_diffusion_avatar_generation_design.json"
future_diffusion_design_path.write_text(json.dumps(future_diffusion_design, indent=2), encoding="utf-8")

print("Saved future diffusion design:", future_diffusion_design_path)
future_diffusion_design

Saved future diffusion design: /workspace/dataset/ksl_project_data/text_to_sign/future_diffusion_avatar_generation_design.json


{'model_name': 'Future diffusion-based sign motion generator',
 'current_status': 'design_only_not_trained',
 'reason_not_trained_now': 'Requires validated KSL motion dataset, signer diversity, and more compute.',
 'input': {'type': 'KSL gloss sequence or healthcare text embedding',
  'example': 'medicine take day two'},
 'output': {'type': 'sign motion sequence',
  'shape_example': '60_frames x 75_joints x 3_coordinates',
  'render_target': '2D skeleton, 3D avatar, or sign video animation'},
 'future_training_data_needed': ['validated KSL healthcare phrase videos',
  'frame-level pose and hand landmarks',
  'gloss labels',
  'healthcare text labels',
  'multiple signers',
  'lighting/background variation'],
 'future_model_pipeline': ['text_to_gloss_model',
  'gloss_embedding',
  'conditional_diffusion_motion_generator',
  'motion_smoothing',
  'avatar_renderer']}

## 14. Web app API design for bidirectional translation

The future web app should expose endpoints for both directions.

## Direction A: sign/video to text/speech

```text
POST /api/sign-to-text/
Input: video frames or uploaded video
Output: recognized gloss + healthcare sentence
```

## Direction B: text/speech to sign/avatar

```text
POST /api/text-to-sign/
Input: doctor text
Output: KSL gloss + sign asset sequence/avatar motion
```

This notebook prepares the second endpoint.

In [15]:
api_design = {
    "sign_to_text_endpoint": {
        "method": "POST",
        "path": "/api/sign-to-text/",
        "input": ["camera_frames", "uploaded_video"],
        "processing": [
            "extract_landmarks",
            "run_recognition_model",
            "generate_gloss",
            "translate_gloss_to_healthcare_text",
            "optional_text_to_speech",
        ],
        "output": {
            "recognized_gloss": "pain where",
            "healthcare_text": "Where is the pain?",
            "confidence": 0.85,
        },
    },
    "text_to_sign_endpoint": {
        "method": "POST",
        "path": "/api/text-to-sign/",
        "input": {
            "text": "Take this medicine twice a day."
        },
        "processing": [
            "normalize_text",
            "map_text_to_gloss",
            "map_gloss_to_sign_assets",
            "future_avatar_or_diffusion_generation",
        ],
        "output": {
            "gloss": "medicine take day two",
            "sign_assets": [
                "assets/signs/medicine.mp4",
                "assets/signs/take.mp4",
                "assets/signs/day.mp4",
                "assets/signs/two.mp4",
            ],
        },
    },
}

api_design_path = TEXT_TO_SIGN_DIR / "bidirectional_webapp_api_design.json"
api_design_path.write_text(json.dumps(api_design, indent=2), encoding="utf-8")

print("Saved API design:", api_design_path)
api_design

Saved API design: /workspace/dataset/ksl_project_data/text_to_sign/bidirectional_webapp_api_design.json


{'sign_to_text_endpoint': {'method': 'POST',
  'path': '/api/sign-to-text/',
  'input': ['camera_frames', 'uploaded_video'],
  'processing': ['extract_landmarks',
   'run_recognition_model',
   'generate_gloss',
   'translate_gloss_to_healthcare_text',
   'optional_text_to_speech'],
  'output': {'recognized_gloss': 'pain where',
   'healthcare_text': 'Where is the pain?',
   'confidence': 0.85}},
 'text_to_sign_endpoint': {'method': 'POST',
  'path': '/api/text-to-sign/',
  'input': {'text': 'Take this medicine twice a day.'},
  'processing': ['normalize_text',
   'map_text_to_gloss',
   'map_gloss_to_sign_assets',
   'future_avatar_or_diffusion_generation'],
  'output': {'gloss': 'medicine take day two',
   'sign_assets': ['assets/signs/medicine.mp4',
    'assets/signs/take.mp4',
    'assets/signs/day.mp4',
    'assets/signs/two.mp4']}}}

## 15. Save text-to-sign mapping model

The mapping model is saved as JSON.

This can be loaded by a future FastAPI/Django backend.

In [16]:
text_to_sign_mapping = {}

for _, row in phrase_df.iterrows():
    text_key = normalize_text(row["healthcare_text"])

    text_to_sign_mapping[text_key] = {
        "healthcare_text": row["healthcare_text"],
        "gloss": row["gloss"],
        "domain": row["domain"],
        "priority": row["priority"],
        "phrase_id": int(row["id"]),
        "asset_sequence": gloss_to_asset_sequence(row["gloss"]),
    }

mapping_path = MODEL_DIR / "text_to_sign_phrase_mapping.json"
mapping_path.write_text(json.dumps(text_to_sign_mapping, indent=2), encoding="utf-8")

print("Saved text-to-sign mapping:", mapping_path)

Saved text-to-sign mapping: /workspace/dataset/ksl_project_data/models/text_to_sign_phrase_mapping.json


## 16. Create metadata summary

This records the models/components used in this notebook.

In [17]:
metadata = {
    "notebook": "08_text_to_sign_generation_design.ipynb",
    "stage": "Text-to-sign generation design",
    "direction": "healthcare worker text/speech to KSL gloss/sign/avatar output",
    "current_models_used": [
        {
            "name": "ExactTextToGlossMapper",
            "type": "rule-based dictionary mapper",
            "purpose": "Maps known healthcare text exactly to KSL-style gloss",
        },
        {
            "name": "FuzzyTextToGlossMapper",
            "type": "token-overlap fuzzy mapper",
            "purpose": "Maps similar healthcare text to the closest known gloss phrase",
        },
        {
            "name": "SignAssetDictionary",
            "type": "asset lookup model",
            "purpose": "Maps gloss tokens to sign video/avatar asset placeholders",
        },
    ],
    "future_models_designed": [
        {
            "name": "mT5/T5 text-to-gloss model",
            "purpose": "Generalize healthcare text to KSL gloss beyond fixed phrase matching",
        },
        {
            "name": "Diffusion-based sign motion generator",
            "purpose": "Generate sign/avatar motion sequences from gloss or text embeddings",
        },
    ],
    "generated_files": {
        "asset_dictionary": str(asset_dictionary_path),
        "demo_outputs": str(demo_output_path),
        "text_to_gloss_dataset": str(text_to_gloss_dataset_path),
        "future_diffusion_design": str(future_diffusion_design_path),
        "api_design": str(api_design_path),
        "text_to_sign_mapping": str(mapping_path),
    },
    "created_at": datetime.now().isoformat(),
}

metadata_path = TEXT_TO_SIGN_DIR / "text_to_sign_generation_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved metadata:", metadata_path)
metadata

Saved metadata: /workspace/dataset/ksl_project_data/text_to_sign/text_to_sign_generation_metadata.json


{'notebook': '08_text_to_sign_generation_design.ipynb',
 'stage': 'Text-to-sign generation design',
 'direction': 'healthcare worker text/speech to KSL gloss/sign/avatar output',
 'current_models_used': [{'name': 'ExactTextToGlossMapper',
   'type': 'rule-based dictionary mapper',
   'purpose': 'Maps known healthcare text exactly to KSL-style gloss'},
  {'name': 'FuzzyTextToGlossMapper',
   'type': 'token-overlap fuzzy mapper',
   'purpose': 'Maps similar healthcare text to the closest known gloss phrase'},
  {'name': 'SignAssetDictionary',
   'type': 'asset lookup model',
   'purpose': 'Maps gloss tokens to sign video/avatar asset placeholders'}],
 'future_models_designed': [{'name': 'mT5/T5 text-to-gloss model',
   'purpose': 'Generalize healthcare text to KSL gloss beyond fixed phrase matching'},
  {'name': 'Diffusion-based sign motion generator',
   'purpose': 'Generate sign/avatar motion sequences from gloss or text embeddings'}],
 'generated_files': {'asset_dictionary': '/workspa

## 17. Thesis interpretation

You can use this paragraph:

> A text-to-sign generation design was developed to support the reverse communication direction from healthcare workers to deaf or hard-of-hearing patients. The baseline approach uses rule-based and fuzzy text-to-gloss mapping to convert known healthcare phrases into draft KSL-style gloss sequences. The glosses are then mapped to sign asset placeholders, enabling a practical minimum viable prototype using sign clips or avatar motion files. The notebook also defines the future design for mT5/T5-based text-to-gloss translation and diffusion-based sign motion generation. This staged approach allows the framework to support immediate phrase-based communication while providing a roadmap toward advanced generative sign language output.

## What this notebook achieved

```text
doctor healthcare text
→ text-to-gloss mapping
→ sign asset sequence
→ future diffusion/avatar design
```

## What comes next

The next notebook should be:

```text
09_realtime_webapp_architecture_and_api_design.ipynb
```

That notebook will design the full web application:

```text
camera/video input
speech input
text input
model serving API
real-time WebSocket flow
avatar/sign output
text and speech output
```

After that, you can start building the actual web app using:

```text
React/Next.js frontend + FastAPI/Django backend + model service
```